# Advanced Neural Architectures for Touch Detection (`model_advanced.ipynb`)

This notebook implements advanced deep learning architectures specifically suited for short spatial-temporal velocity features ($4 \text{ timesteps} \times 8 \text{ features} = 32 \text{ dimensions}$):

1. **ResNet1D (Residual 1D CNN with Skip Connections)**: Allows direct gradient flow across transition steps without information bottleneck.
2. **TouchAttentionNet (Multi-Head Self-Attention)**: Uses self-attention to capture dynamic inter-joint velocity correlations (e.g., DIP deceleration vs MCP movement).

## 1. Imports & Hyperparameters Setup

In [6]:
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Set random seeds
RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

# Hyperparameters
SEQ_LEN = 4          # 4 velocity transition steps
FEATURE_DIM = 8      # 2 wrist vels + 6 finger joint vels
BATCH_SIZE = 32
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-4
EPOCHS = 40

# Setup device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## 2. Load & Prepare Datasets (`training_data.csv` & `test_data.csv`)

In [7]:
def extract_features_and_labels(csv_path):
    df = pd.read_csv(csv_path)
    
    vel_cols_step = []
    for v in range(1, 5):
        cols = [
            f"wrist{v}_vx", f"wrist{v}_vy",
            f"mcp{v}_vx", f"mcp{v}_vy",
            f"pip{v}_vx", f"pip{v}_vy",
            f"dip{v}_vx", f"dip{v}_vy"
        ]
        vel_cols_step.append(cols)
        
    n_samples = len(df)
    X = np.zeros((n_samples, 4, 8), dtype=np.float32)
    
    for step_idx in range(4):
        cols = vel_cols_step[step_idx]
        X[:, step_idx, :] = df[cols].fillna(0.0).values.astype(np.float32)
        
    target_col = "touch_finger" if "touch_finger" in df.columns else "touch"
    y = df[target_col].astype(str).str.strip().str.lower().isin(["1", "true", "t", "yes", "y"]).values.astype(np.float32)
    y = y.reshape(-1, 1)
    
    return X, y

TRAIN_CSV = "./data/training_data.csv"
TEST_CSV = "./data/test_data.csv"

X_train_np, y_train_np = extract_features_and_labels(TRAIN_CSV)
X_test_np, y_test_np = extract_features_and_labels(TEST_CSV)

# Scaler
scaler = StandardScaler()
N_tr, T, C = X_train_np.shape
N_te, _, _ = X_test_np.shape

X_train_flat = X_train_np.reshape(N_tr, -1)
X_test_flat = X_test_np.reshape(N_te, -1)

X_train_scaled_flat = scaler.fit_transform(X_train_flat)
X_test_scaled_flat = scaler.transform(X_test_flat)

X_train_scaled = X_train_scaled_flat.reshape(N_tr, T, C)
X_test_scaled = X_test_scaled_flat.reshape(N_te, T, C)

X_train_tensor = torch.from_numpy(X_train_scaled).type(torch.float32)
y_train_tensor = torch.from_numpy(y_train_np).type(torch.float32)
X_test_tensor = torch.from_numpy(X_test_scaled).type(torch.float32)
y_test_tensor = torch.from_numpy(y_test_np).type(torch.float32)

print(f"X_train shape: {X_train_tensor.shape}, y_train shape: {y_train_tensor.shape}")
print(f"X_test shape:  {X_test_tensor.shape},  y_test shape:  {y_test_tensor.shape}")

X_train shape: torch.Size([1791, 4, 8]), y_train shape: torch.Size([1791, 1])
X_test shape:  torch.Size([315, 4, 8]),  y_test shape:  torch.Size([315, 1])


## 3. Model Architecture 1: 1D Residual Network (ResNet1D)

Combines residual skip connections (`x + conv(x)`) with 1D Convolutions to eliminate vanishing gradients.

In [8]:
class ResBlock1D(nn.Module):
    def __init__(self, channels: int, dropout: float = 0.2):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv1d(channels, channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(channels),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(channels, channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(channels)
        )
        self.relu = nn.ReLU()
        
    def forward(self, x):
        return self.relu(x + self.block(x))

class FingerTouchResNet1D(nn.Module):
    def __init__(self, in_channels: int = 8, hidden_dim: int = 64, dropout: float = 0.2):
        super().__init__()
        self.input_conv = nn.Sequential(
            nn.Conv1d(in_channels, hidden_dim, kernel_size=3, padding=1),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU()
        )
        self.res1 = ResBlock1D(hidden_dim, dropout)
        self.res2 = ResBlock1D(hidden_dim, dropout)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(hidden_dim, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1)
        )
        
    def forward(self, x):
        x_conv = x.permute(0, 2, 1)
        out = self.input_conv(x_conv)
        out = self.res1(out)
        out = self.res2(out)
        out = self.pool(out)
        return self.classifier(out)

resnet1d = FingerTouchResNet1D(in_channels=8, hidden_dim=64).to(device)
print(resnet1d)

FingerTouchResNet1D(
  (input_conv): Sequential(
    (0): Conv1d(8, 64, kernel_size=(3,), stride=(1,), padding=(1,))
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
  )
  (res1): ResBlock1D(
    (block): Sequential(
      (0): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(1,))
      (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): Dropout(p=0.2, inplace=False)
      (4): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(1,))
      (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (relu): ReLU()
  )
  (res2): ResBlock1D(
    (block): Sequential(
      (0): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(1,))
      (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): Dropout(p=0.2, inplace=False)
      (4): Conv1d(64, 64, kernel_size=(3,), str

## 4. Model Architecture 2: Transformer / Self-Attention Network

Uses Multi-Head Self-Attention to dynamically weigh joint interactions (e.g. wrist vs DIP velocity correlation).

In [9]:
class TouchAttentionNet(nn.Module):
    def __init__(self, input_dim: int = 8, embed_dim: int = 32, num_heads: int = 4, dropout: float = 0.2):
        super().__init__()
        self.embedding = nn.Linear(input_dim, embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, embed_dim)
        )
        self.norm2 = nn.LayerNorm(embed_dim)
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )
        
    def forward(self, x):
        emb = self.embedding(x)
        attn_out, _ = self.attn(emb, emb, emb)
        x_attn = self.norm1(emb + attn_out)
        ffn_out = self.ffn(x_attn)
        x_out = self.norm2(x_attn + ffn_out)
        pooled = x_out.mean(dim=1)
        return self.classifier(pooled)

attn_net = TouchAttentionNet(input_dim=8, embed_dim=32, num_heads=4).to(device)
print(attn_net)

TouchAttentionNet(
  (embedding): Linear(in_features=8, out_features=32, bias=True)
  (attn): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=32, out_features=32, bias=True)
  )
  (norm1): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
  (ffn): Sequential(
    (0): Linear(in_features=32, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=64, out_features=32, bias=True)
  )
  (norm2): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
  (classifier): Sequential(
    (0): Linear(in_features=32, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=1, bias=True)
  )
)


## 5. Training PyTorch Neural Architectures

In [10]:
train_dataset = DataLoader(list(zip(X_train_tensor, y_train_tensor)), batch_size=32, shuffle=True)
test_dataset = DataLoader(list(zip(X_test_tensor, y_test_tensor)), batch_size=32, shuffle=False)

def train_model(model, epochs=40, lr=0.001):
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
    
    train_losses, test_losses = [], []
    train_accs, test_accs = [], []
    
    def accuracy_fn(y_true, y_pred):
        return (torch.eq(y_true, y_pred).sum().item() / len(y_pred)) * 100.0
        
    for epoch in range(1, epochs + 1):
        model.train()
        tr_loss, tr_acc = 0.0, 0.0
        for X_b, y_b in train_dataset:
            X_b, y_b = X_b.to(device), y_b.to(device)
            logits = model(X_b)
            loss = loss_fn(logits, y_b)
            preds = torch.round(torch.sigmoid(logits))
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            tr_loss += loss.item() * len(X_b)
            tr_acc += accuracy_fn(y_b, preds) * len(X_b) / 100.0
            
        model.eval()
        te_loss, te_acc = 0.0, 0.0
        with torch.inference_mode():
            for X_b, y_b in test_dataset:
                X_b, y_b = X_b.to(device), y_b.to(device)
                logits = model(X_b)
                loss = loss_fn(logits, y_b)
                preds = torch.round(torch.sigmoid(logits))
                te_loss += loss.item() * len(X_b)
                te_acc += accuracy_fn(y_b, preds) * len(X_b) / 100.0
                
        tr_acc = (tr_acc / len(X_train_tensor)) * 100.0
        te_acc = (te_acc / len(X_test_tensor)) * 100.0
        te_loss /= len(X_test_tensor)
        scheduler.step(te_loss)
        
        if epoch % 5 == 0 or epoch == 1:
            print(f"Epoch: {epoch:02d} | Train Acc: {tr_acc:.2f}% | Test Loss: {te_loss:.4f} | Test Acc: {te_acc:.2f}%")
            
    return te_acc

print("\n=== Training ResNet1D ===")
resnet_acc = train_model(resnet1d, epochs=EPOCHS)

print("\n=== Training TouchAttentionNet ===")
attn_acc = train_model(attn_net, epochs=EPOCHS)


=== Training ResNet1D ===
Epoch: 01 | Train Acc: 82.24% | Test Loss: 0.2688 | Test Acc: 89.84%
Epoch: 05 | Train Acc: 90.06% | Test Loss: 0.2599 | Test Acc: 89.84%
Epoch: 10 | Train Acc: 93.08% | Test Loss: 0.2347 | Test Acc: 91.43%
Epoch: 15 | Train Acc: 94.30% | Test Loss: 0.2993 | Test Acc: 90.79%
Epoch: 20 | Train Acc: 95.20% | Test Loss: 0.2724 | Test Acc: 92.06%
Epoch: 25 | Train Acc: 94.64% | Test Loss: 0.3044 | Test Acc: 91.75%
Epoch: 30 | Train Acc: 95.59% | Test Loss: 0.2850 | Test Acc: 92.70%
Epoch: 35 | Train Acc: 94.75% | Test Loss: 0.3082 | Test Acc: 91.11%
Epoch: 40 | Train Acc: 95.76% | Test Loss: 0.2943 | Test Acc: 92.06%

=== Training TouchAttentionNet ===
Epoch: 01 | Train Acc: 72.59% | Test Loss: 0.3560 | Test Acc: 89.21%
Epoch: 05 | Train Acc: 89.22% | Test Loss: 0.2493 | Test Acc: 90.79%
Epoch: 10 | Train Acc: 90.51% | Test Loss: 0.2446 | Test Acc: 90.79%
Epoch: 15 | Train Acc: 90.68% | Test Loss: 0.2564 | Test Acc: 90.48%
Epoch: 20 | Train Acc: 91.01% | Test Los

## 6. Model Performance Summary & Saving Models

In [6]:
print("\n" + "="*60)
print(" NEURAL ARCHITECTURES ACCURACY COMPARISON")
print("="*60)
print(f"  ResNet1D (Skip CNN):   {resnet_acc:.2f}%")
print(f"  TouchAttentionNet:     {attn_acc:.2f}%")
print("="*60)

torch.save(resnet1d.state_dict(), "finger_touch_resnet1d.pth")
torch.save(attn_net.state_dict(), "finger_touch_attention.pth")
print("Saved model weights to 'finger_touch_resnet1d.pth' and 'finger_touch_attention.pth'.")